# 1단계: MVTec AD 재현

PatchCore(원본 코드)와 PaDiM(직접 구현)이 원 논문 값과 맞는지 확인한다. 설정 4개 x 15 부류 x 시드 3개, 모두 180번 돈다.

| 설정 | 이미지 AUROC | 픽셀 AUROC | PRO |
|---|---:|---:|---:|
| PatchCore-10%, WRN-50-2 (2,3) | 99.0 | 98.1 | 93.5 |
| PatchCore-1%, WRN-50-2 (2,3) | 99.0 | 98.0 | 93.1 |
| PaDiM, WRN-50-2 (차원 550) | 95.3 | 97.5 | 92.1 |
| PaDiM, ResNet18 (차원 100) | 논문에 없음 | 96.7 | 90.1 |

판정 폭은 이미지 AUROC ±0.5, 픽셀 AUROC와 PRO ±1.0이다. 맞지 않으면 2단계로 가지 않는다.

**실행 전 준비 둘.**
1. 런타임 유형을 GPU(L4)로 바꾼다.
2. 레포가 비공개이므로, 왼쪽 열쇠 아이콘(보안 비밀)에 `GITHUB_TOKEN`을 넣고 이 노트북에 접근을 허용한다. 레포 읽기 권한만 있는 토큰이면 된다.

끊겨도 된다. 결과는 부류 하나가 끝날 때마다 Drive의 CSV에 적고, 같은 셀을 다시 돌리면 끝난 줄을 건너뛴다.

## 1. 코드와 패키지

In [ ]:
import os, subprocess, sys

REPO_DIR = "/content/beyond-mvtec"
UPSTREAM_DIR = "/content/patchcore-inspection"
UPSTREAM_COMMIT = "fcaa92f124fb1ad74a7acf56726decd4b27cbcad"

if not os.path.isdir(REPO_DIR):
    from google.colab import userdata
    token = userdata.get("GITHUB_TOKEN")
    subprocess.run(["git", "clone", "-q", f"https://{token}@github.com/chlee1027/beyond-mvtec.git", REPO_DIR], check=True)
    # 토큰이 설정 파일에 남지 않게 지운다
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", "https://github.com/chlee1027/beyond-mvtec.git"], check=True)
if not os.path.isdir(UPSTREAM_DIR):
    subprocess.run(["git", "clone", "-q", "https://github.com/amazon-science/patchcore-inspection.git", UPSTREAM_DIR], check=True)
subprocess.run(["git", "-C", UPSTREAM_DIR, "checkout", "-q", UPSTREAM_COMMIT], check=True)

!pip uninstall -y -q sympy
!pip install -q sympy==1.13.3 faiss-gpu-cu12 timm click pretrainedmodels scikit-image scikit-learn scipy tqdm pandas pyyaml

sys.path[:0] = [os.path.join(REPO_DIR, "src"), os.path.join(UPSTREAM_DIR, "src")]
COMMIT = subprocess.run(["git", "-C", REPO_DIR, "rev-parse", "--short", "HEAD"], capture_output=True, text=True).stdout.strip()

import faiss, torch, torchvision
assert torch.cuda.is_available(), "GPU 런타임이 아니다"
print(f"레포 {COMMIT} | torch {torch.__version__} | torchvision {torchvision.__version__} | faiss {faiss.__version__}")
print(f"GPU {torch.cuda.get_device_name(0)} | VRAM {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
assert hasattr(faiss, "StandardGpuResources"), "faiss 의 GPU 판이 아니다"

## 2. Drive와 데이터

In [ ]:
import shutil
from google.colab import drive
drive.mount("/content/drive")

MY = "/content/drive/MyDrive"
os.environ["TORCH_HOME"] = f"{MY}/torch_cache"   # 사전학습 가중치를 세션마다 받지 않게 둔다
RESULT_DIR = f"{MY}/beyond_mvtec_results"
os.makedirs(RESULT_DIR, exist_ok=True)

MVTEC_CANDIDATES = [f"{MY}/anormaly_detection/mvtec", f"{MY}/mvtec", f"{MY}/MVTecAD"]
MVTEC_DRIVE = next((p for p in MVTEC_CANDIDATES if os.path.isdir(os.path.join(p, "bottle"))), None)
assert MVTEC_DRIVE, f"MVTec 폴더를 못 찾았다. 후보: {MVTEC_CANDIDATES}"

# Drive 에서 파일을 하나씩 읽으면 느려서 런타임 디스크로 옮긴다
MVTEC_LOCAL = "/content/data/mvtec_ad"
if not os.path.isfile(f"{MVTEC_LOCAL}/.copied"):
    os.makedirs(os.path.dirname(MVTEC_LOCAL), exist_ok=True)
    shutil.copytree(MVTEC_DRIVE, MVTEC_LOCAL, dirs_exist_ok=True)
    open(f"{MVTEC_LOCAL}/.copied", "w").close()
print("Drive:", MVTEC_DRIVE, "-> 로컬:", MVTEC_LOCAL)

In [ ]:
import pandas as pd
from beyond_mvtec import data, experiment as ex

rows = []
for category in data.MVTEC_CATEGORIES:
    split = data.list_mvtec(MVTEC_LOCAL, category)
    rows.append({"category": category, "train": len(split.train), "test": len(split.test),
                 "test_anomaly": sum(s.label for s in split.test)})
table = pd.DataFrame(rows)
display(table)
# 논문의 장수와 맞는지 확인한다
assert table["train"].sum() == 3629 and table["test"].sum() == 1725, (table["train"].sum(), table["test"].sum())
print("학습 3,629장, 시험 1,725장: 논문과 같다")

## 3. 짧은 실행

전체를 돌리기 전에 부류 하나로 GPU 메모리와 faiss GPU가 도는지 본다. 이 값은 재현 판정에 쓰지 않는다.

In [ ]:
config = ex.load_config(f"{REPO_DIR}/configs/experiments.yaml")
PREPROCESS = config["preprocess"]["reproduce"]
SETTINGS = ex.reproduce_settings(config)
OPTIONS = dict(device="cuda", batch_size=8, num_workers=4, faiss_on_gpu=True, commit=COMMIT)

smoke = {s["setting"]: s for s in SETTINGS}
for name in ["padim_wrn50_L123_d550", "patchcore_wrn50_L23_c1"]:
    row = ex.run_one(smoke[name], "mvtec_ad", MVTEC_LOCAL, "bottle", 0, "reproduce", PREPROCESS, **OPTIONS)
    print({k: row[k] for k in ["setting", "image_auroc", "pixel_auroc", "pro", "bank_size", "fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]})

## 4. 재현 실행

시드 0의 이상 지도는 `maps/` 에 남긴다. PRO가 어긋날 때 정의를 바꿔 다시 재기 위해서다. 부류 하나마다 걸린 시간과 GPU 메모리가 CSV에 남는다.

In [ ]:
CSV_PATH = f"{RESULT_DIR}/reproduce_mvtec.csv"
ex.run_grid(SETTINGS, "mvtec_ad", MVTEC_LOCAL, data.MVTEC_CATEGORIES, config["seeds"], "reproduce", PREPROCESS,
            CSV_PATH, save_dir=f"{RESULT_DIR}/maps", save_maps_seeds=(0,), **OPTIONS)

## 5. 판정

In [ ]:
summary = ex.summarize(CSV_PATH)
display(summary.round(3))

tolerance = config["reproduce"]["tolerance"]
verdict = ex.judge_reproduction(summary, SETTINGS, tolerance)
display(verdict)

complete = (verdict["n_seeds"] == 3).all() and (verdict["n_categories"] == 15).all()
judged = verdict[verdict["passed"].notna()]
print("칸이 다 찼나:", complete)
print(f"판정 폭 안: {int((judged['passed'] == True).sum())} / {len(judged)}")
print("다음 단계로 가도 되나:", bool(complete and (judged["passed"] == True).all()))

In [ ]:
# 부류별 값(시드 평균, %). PaDiM 은 논문 표 III 의 부류 값과, PatchCore 는 표 S1 과 맞대어 본다.
# PatchCore 의 부류 값은 원본 코드와 논문이 이미 다르므로(Screw 1.77점) 15 부류 평균 폭을 부류별로 쓰지 않는다.
df = pd.read_csv(CSV_PATH)
for metric in ex.METRICS:
    print(metric)
    display((df.pivot_table(index="category", columns="setting", values=metric, aggfunc="mean") * 100).round(2))
print("시간(초), 설정별 부류 평균")
display(df.groupby("setting")[["fit_sec", "predict_sec", "metric_sec", "peak_gpu_gb"]].mean().round(1))

## 6. 결과를 레포로

`reproduce_mvtec.csv`는 Drive의 `beyond_mvtec_results/`에 있다. 이 파일을 레포의 `results/`에 넣고 커밋한다. 지도(`maps/`)는 레포에 올리지 않는다.